In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import zipfile

# -------------------------------------------------------------------
# 1. Map Windows Drive paths to Colab Paths
# -------------------------------------------------------------------
BASE_DRIVE    = "/content/drive/MyDrive/papers/Sabbir"
WORKSPACE_DIR = os.path.join(BASE_DRIVE, "KidneyStoneAI")
DATASETS_DIR  = os.path.join(BASE_DRIVE, "Datasets")

# Specific files based on your structure
REAL_ZIP      = os.path.join(DATASETS_DIR, "Real Data.zip")

# Output directories inside your workspace
MODELS_DIR    = os.path.join(WORKSPACE_DIR, "models_v2")
CKPT_DIR      = os.path.join(WORKSPACE_DIR, "checkpoints_v2")

# Local runtime extraction path (for fast training)
SPLIT_ROOT    = "/content/ksd1_split_real"

print("--- Step 1: Environment & Path Verification ---")
print(f"Workspace Directory exists: {os.path.exists(WORKSPACE_DIR)}")
print(f"Real Dataset Zip exists:    {os.path.exists(REAL_ZIP)}")

# -------------------------------------------------------------------
# 2. Create Workspace Output Directories
# -------------------------------------------------------------------
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print("Models and Checkpoint directories verified inside Workspace.")

# -------------------------------------------------------------------
# 3. Extract Real Dataset Cleanly
# -------------------------------------------------------------------
if not os.path.exists(SPLIT_ROOT):
    print("\nExtracting Real Dataset to local runtime...")
    try:
        with zipfile.ZipFile(REAL_ZIP, 'r') as zip_ref:
            zip_ref.extractall(SPLIT_ROOT)
        print("✅ Extraction complete!")
    except FileNotFoundError:
        print(f"❌ ERROR: Could not find {REAL_ZIP}. Please check the filename.")
else:
    print("\nDataset already extracted locally.")

# Verify the extracted structure
if os.path.exists(SPLIT_ROOT):
    print("\nVerifying Dataset Splits:")
    # Handle case where the zip might extract into a nested folder (e.g., 'Real Data/train')
    extracted_items = os.listdir(SPLIT_ROOT)
    if 'train' not in extracted_items and len(extracted_items) == 1:
        # If there's a parent folder inside the zip
        SPLIT_ROOT = os.path.join(SPLIT_ROOT, extracted_items[0])
        print(f"Adjusted SPLIT_ROOT to: {SPLIT_ROOT}")

    for split in ['train', 'val', 'test']:
        p = os.path.join(SPLIT_ROOT, split)
        if os.path.exists(p):
            print(f" -> {split} classes found: {os.listdir(p)}")
        else:
            print(f" -> ⚠️ Warning: {split} folder not found!")

Mounted at /content/drive
--- Step 1: Environment & Path Verification ---
Workspace Directory exists: True
Real Dataset Zip exists:    True
Models and Checkpoint directories verified inside Workspace.

Extracting Real Dataset to local runtime...
✅ Extraction complete!

Verifying Dataset Splits:
 -> train classes found: ['stone', 'normal']
 -> val classes found: ['stone', 'normal']
 -> test classes found: ['stone', 'normal']


In [3]:
import os, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, classification_report

print(f"Torch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

# -------------------------------------------------------------------
# 1. Path Definitions (Carried over from Step 1)
# -------------------------------------------------------------------
WORKSPACE_DIR = "/content/drive/MyDrive/papers/Sabbir/KidneyStoneAI"
MODELS_DIR    = os.path.join(WORKSPACE_DIR, "models_v2")
SPLIT_ROOT    = "/content/ksd1_split_real"
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "ksd1_mamba_real_best_v2.pth")

# -------------------------------------------------------------------
# 2. Pure PyTorch Vision Mamba Architecture
# -------------------------------------------------------------------
class PureMambaBlock(nn.Module):
    """Pure PyTorch approximation of Mamba Block (Selective SSM)"""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner, out_channels=self.d_inner,
            kernel_size=d_conv, groups=self.d_inner, padding=d_conv - 1
        )

        self.x_proj = nn.Linear(self.d_inner, self.d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        x_in = x_in.transpose(1, 2)
        x_in = self.conv1d(x_in)[:, :, :L].transpose(1, 2)
        x_in = F.silu(x_in)

        x_dbl = self.x_proj(x_in)
        delta, B_mat, C_mat = torch.split(x_dbl, [1, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))

        A = -torch.exp(self.A_log)
        y = torch.zeros_like(x_in)
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device)

        for t in range(L):
            delta_t = delta[:, t, :].unsqueeze(-1)
            B_t = B_mat[:, t, :].unsqueeze(1)
            C_t = C_mat[:, t, :].unsqueeze(1)
            x_t = x_in[:, t, :].unsqueeze(-1)

            delta_A = torch.exp(delta_t * A)
            delta_B = (torch.exp(delta_t * A) - 1) / (A + 1e-6) * B_t
            h = delta_A * h + delta_B * x_t
            y[:, t, :] = torch.sum(h * C_t, dim=-1) + self.D * x_in[:, t, :]

        return self.out_proj(y * F.silu(z))

class VisionMamba(nn.Module):
    def __init__(self, img_size=128, patch_size=8, in_chans=3, num_classes=2, embed_dim=192, depth=12):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.blocks = nn.ModuleList([PureMambaBlock(d_model=embed_dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)

        # Matches KSD1_MLPHead from your previous models
        self.head = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(), nn.Dropout(p=0.4), nn.Linear(embed_dim // 2, num_classes)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1) + self.pos_embed
        for blk in self.blocks: x = x + blk(F.layer_norm(x, x.shape[1:]))
        return self.head(self.norm(x)[:, 0])

# -------------------------------------------------------------------
# 3. Data Loaders (UPDATED FOR MEMORY EFFICIENCY)
# -------------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((128, 128)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((128, 128)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# REDUCED BATCH SIZE TO 4 TO PREVENT OOM
MICRO_BATCH = 4
train_loader = DataLoader(ImageFolder(os.path.join(SPLIT_ROOT, "train"), transform=train_transform), batch_size=MICRO_BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(ImageFolder(os.path.join(SPLIT_ROOT, "val"), transform=val_transform), batch_size=MICRO_BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ImageFolder(os.path.join(SPLIT_ROOT, "test"), transform=val_transform), batch_size=MICRO_BATCH, shuffle=False, num_workers=2, pin_memory=True)

# -------------------------------------------------------------------
# 4. Training Loop (UPDATED WITH GRADIENT ACCUMULATION)
# -------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionMamba().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler = GradScaler()

ACCUMULATION_STEPS = 32 // MICRO_BATCH # 32 / 4 = 8 steps. Maintains effective batch size of 32!
best_val_acc = 0.0

# Clear any residual memory from the crash
torch.cuda.empty_cache()

print(f"\n🚀 Starting Training: Vision Mamba (REAL) | Micro-batch: {MICRO_BATCH} | Effective Batch: 32...")
for epoch in range(15):
    model.train()
    t0 = time.time()
    train_loss, train_preds, train_targets = 0.0, [], []

    optimizer.zero_grad() # Zero gradients at the start of the epoch

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        with autocast(device_type='cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss = loss / ACCUMULATION_STEPS # Normalize loss for accumulation

        scaler.scale(loss).backward()

        # Only step the optimizer every ACCUMULATION_STEPS
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS * inputs.size(0)
        train_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    scheduler.step()

    # Validation loop (also adjusted micro-batch handling)
    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            with autocast(device_type='cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            val_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    tl, ta = train_loss / len(train_loader.dataset), accuracy_score(train_targets, train_preds)
    vl, va = val_loss / len(val_loader.dataset), accuracy_score(val_targets, val_preds)

    print(f"Epoch {epoch+1:02d} | TL={tl:.4f} TA={ta:.4f} | VL={vl:.4f} VA={va:.4f} | {time.time()-t0:.1f}s")
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("   -> 🌟 Best model saved!")

# -------------------------------------------------------------------
# 5. Testing & Evaluation
# -------------------------------------------------------------------
print("\n📊 Loading Best Model for Testing...")
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()
test_preds, test_targets, test_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with autocast(device_type='cuda'):
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
        test_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        test_targets.extend(labels.cpu().numpy())
        test_probs.extend(probs[:, 1].cpu().numpy())

print("\n==============================")
print(" VISION MAMBA (REAL) METRICS")
print("==============================")
print(f"Accuracy : {accuracy_score(test_targets, test_preds):.4f}")
print(f"Recall   : {recall_score(test_targets, test_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(test_targets, test_probs):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(test_targets, test_preds))
print("\nReport:\n", classification_report(test_targets, test_preds, target_names=["normal", "stone"]))

Torch Version: 2.11.0+cu128 | CUDA Available: True

🚀 Starting Training: Vision Mamba (REAL) | Micro-batch: 4 | Effective Batch: 32...


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 11.01 GiB is allocated by PyTorch, and 3.42 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)